This notebook was created by Donna Faith Go.

In [1]:
# import sys
# !{sys.executable} -m pip install -qq -r requirements.txt

In [2]:
# standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import re

# knn imputer
from sklearn.impute import KNNImputer

# ignore warnings
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

## Applying Granger Causality

## Access Data

In [3]:
filepath = r'SP500 comapnies list.pkl'
with open(filepath, 'rb') as f:
    stocks_info = pickle.load(f)
stocks_info.set_index('Security', inplace=True)
stocks_info

,Symbol,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
Security,,,,,,,
3M,MMM,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
A. O. Smith,AOS,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
Abbott Laboratories,ABT,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
AbbVie,ABBV,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
Accenture,ACN,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...
Xylem Inc.,XYL,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
Yum! Brands,YUM,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
Zebra Technologies,ZBRA,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969


In [4]:
directory = r'data'
closing_dict = {}

for name in sorted(os.listdir(directory)):
    if name == '.ipynb_checkpoints':
        continue

    # getting the ticker 
    firm = name.split('.pkl')[0]
    if firm in stocks_info.index:
        symbol = stocks_info.loc[firm, 'Symbol']

    # getting the file
    filepath = os.path.join(directory, name)
    data = pd.read_pickle(filepath)
    data.columns = data.columns.get_level_values(0)
    data = data[['Close', 'High', 'Low', 'Open', 'Volume']]
    data.columns.name = None

    # getting the closing prices
    closing = data.loc[:, 'Close']
    closing = closing[~closing.index.duplicated(keep='first')]
    closing_dict[symbol] = closing

# creating a dataframe
closing_df = pd.concat(closing_dict, axis=1)
closing_df.sort_index(inplace=True)

# dropping columns that have all nulls
closing_df.dropna(how='all', axis=1, inplace=True)
original_closing = closing_df.copy()

In [5]:
closing_df.head(10)

,MMM,AOS,AES,APA,T,ABBV,ABT,ACN,ADBE,AMD,...,WTW,WDAY,WYNN,XEL,XYL,YUM,ZBRA,ZBH,ZTS,EBAY
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,19.257389,2.254067,23.135981,10.113912,6.250895,NaN,8.093055,NaN,16.274677,15.5000,...,NaN,NaN,NaN,6.577461,NaN,4.503591,25.027779,NaN,NaN,6.613386
2000-01-04,18.492201,2.221588,22.218521,9.669347,5.885149,NaN,7.861829,NaN,14.909399,14.6250,...,NaN,NaN,NaN,6.728915,NaN,4.413068,24.666668,NaN,NaN,5.993016
2000-01-05,19.027836,2.215093,22.457857,9.947192,5.976584,NaN,7.847377,NaN,15.204176,15.0000,...,NaN,NaN,NaN,6.988552,NaN,4.435700,25.138889,NaN,NaN,6.393916
2000-01-06,20.558224,2.182613,22.637363,10.891903,5.860783,NaN,8.121960,NaN,15.328288,16.0000,...,NaN,NaN,NaN,6.923641,NaN,4.397984,23.777779,NaN,NaN,6.314907
2000-01-07,20.966341,2.273555,23.076149,10.854856,5.911020,NaN,8.208676,NaN,16.072985,16.2500,...,NaN,NaN,NaN,6.923641,NaN,4.299915,23.513889,NaN,NaN,6.309053
2000-01-10,20.864305,2.299538,24.113276,10.428813,5.994746,NaN,8.150867,NaN,16.693560,17.5000,...,NaN,NaN,NaN,6.923641,NaN,4.473421,24.305555,NaN,NaN,6.660207
2000-01-11,20.507210,2.273555,24.232958,10.465861,5.911020,NaN,8.035254,NaN,15.545492,17.2500,...,NaN,NaN,NaN,6.923641,NaN,4.443243,24.472221,NaN,NaN,6.516820
2000-01-12,20.558224,2.195606,24.731567,10.410286,5.601234,NaN,7.915000,NaN,15.467917,18.1250,...,NaN,NaN,NaN,7.075095,NaN,4.397984,23.861111,NaN,NaN,6.104214
2000-01-13,20.558224,2.228083,25.150408,10.632575,5.467273,NaN,7.842385,NaN,16.290184,18.8750,...,NaN,NaN,NaN,7.053460,NaN,4.465875,23.944445,NaN,NaN,6.452442


## Data Preprocessing

### Log Returns

In [6]:
# get log returns
log_returns = np.log(closing_df / closing_df.shift(1))
log_returns.dropna(how='all', axis=0, inplace=True)

### K-NN Imputer

In [7]:
# create model
imputer = KNNImputer(n_neighbors=5)
transformed_array = imputer.fit_transform(log_returns)

# create dataframe
log_returns_transformed = pd.DataFrame(
    transformed_array, 
    columns=log_returns.columns,
    index=log_returns.index
)

,MMM,AOS,AES,APA,T,ABBV,ABT,ACN,ADBE,AMD,...,WTW,WDAY,WYNN,XEL,XYL,YUM,ZBRA,ZBH,ZTS,EBAY
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-04,-0.040546,-0.014514,-0.040463,-0.044951,-0.060293,-0.047986,-0.028987,-0.043908,-0.087619,-0.058108,...,-0.016208,-0.038481,-0.044453,0.022765,-0.025964,-0.020305,-0.014533,-0.024847,-0.029625,-0.098501
2000-01-05,0.028554,-0.002928,0.010714,0.028330,0.015417,0.002095,-0.001840,0.004753,0.019578,0.025318,...,0.007332,0.005072,-0.000951,0.037859,0.008842,0.005115,0.018963,0.006503,0.006926,0.064752
2000-01-06,0.077358,-0.014771,0.007961,0.090729,-0.019566,-0.009589,0.034392,-0.003899,0.008130,0.064539,...,0.005169,-0.023491,0.003495,-0.009332,0.006380,-0.008539,-0.055665,0.006937,-0.004876,-0.012434
2000-01-07,0.019657,0.040822,0.019198,-0.003407,0.008535,0.030152,0.010620,0.032197,0.047440,0.015504,...,0.007538,0.038745,0.011205,0.000000,0.027753,-0.022551,-0.011160,0.026553,0.027329,-0.000927
2000-01-10,-0.004879,0.011364,0.043963,-0.040040,0.014065,0.007978,-0.007067,0.008697,0.037883,0.074108,...,0.003850,0.030715,0.010249,0.000000,0.011633,0.039558,0.033114,0.014928,0.012598,0.054165


## Shock Periods